In [1]:
# Exercise 3

import os

project_dir = "nos-flask"
os.makedirs(project_dir, exist_ok=True)
print(f"Folder '{project_dir}' created or already exists.")

# -------------------------------
# Step 1: Create app.py
# -------------------------------
app_py_content = """
from flask import Flask, jsonify
import os, socket, datetime

app = Flask(__name__)

@app.route('/')
def index():
    return jsonify({
        'message' : 'Hello from inside a Docker container! 🐳',
        'hostname' : socket.gethostname(),
        'time' : datetime.datetime.utcnow().isoformat(),
        'version' : os.getenv('APP_VERSION', '1.0'),
    })

@app.route('/health')
def health():
    return jsonify({'status': 'ok'}), 200

@app.route('/info')
def info():
    return jsonify(dict(os.environ))

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
"""

with open(os.path.join(project_dir, "app.py"), "w", encoding="utf-8") as f:
    f.write(app_py_content)
print("app.py created with UTF-8 encoding.")

# -------------------------------
# Step 2: Create requirements.txt
# -------------------------------
requirements_content = """flask==3.0.0
gunicorn==21.2.0
"""

with open(os.path.join(project_dir, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write(requirements_content)
print("requirements.txt created.")

# -------------------------------
# Step 3: Create Dockerfile
# -------------------------------
dockerfile_content = """
FROM python:3.11-slim

LABEL maintainer='nos-module@university.ac.uk'
LABEL description='NOS Module - Flask demo container'

ARG VERSION=1.0
ENV APP_VERSION=${VERSION}

WORKDIR /app

# Copy dependencies first for layer caching
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app source
COPY app.py .

EXPOSE 5000

# Create non-root user
RUN useradd --create-home appuser
USER appuser

CMD ["gunicorn", "--bind", "0.0.0.0:5000", "--workers", "2", "app:app"]
"""

with open(os.path.join(project_dir, "Dockerfile"), "w", encoding="utf-8") as f:
    f.write(dockerfile_content)
print("Dockerfile created.")

# -------------------------------
# Step 4: Clean up old containers
# -------------------------------
!docker rm -f flask1 flask2 flask3 2>nul || echo "No old containers to remove."

# -------------------------------
# Step 5: Build Docker images
# -------------------------------
print("\nBuilding nos-flask:1.0...")
!docker build -t nos-flask:1.0 {project_dir}

print("\nBuilding nos-flask:1.1 (small change caching demo)...")
!docker build -t nos-flask:1.1 {project_dir}

print("\nBuilding nos-flask:2.0 (VERSION=2.0)...")
!docker build --build-arg VERSION=2.0 -t nos-flask:2.0 {project_dir}

print("\nBuilding nos-flask:2.1 (includes /info route)...")
!docker build -t nos-flask:2.1 {project_dir}

# -------------------------------
# Step 6: Run containers safely
# -------------------------------
!docker run -d -p 5000:5000 --name flask1 nos-flask:2.1
!docker run -d -p 5001:5000 --name flask2 nos-flask:2.0

# -------------------------------
# Step 7: Test endpoints
# -------------------------------
print("\nTesting Flask container 1 (/info route):")
!curl http://localhost:5000/info

print("\nTesting Flask container 2 (version=2.0):")
!curl http://localhost:5001

# -------------------------------
# Step 8: Inspect running container
# -------------------------------
print("\nProcesses in flask1 container:")
!docker exec -it flask1 sh -c "ps aux"

print("\nUser info in flask1 container:")
!docker exec -it flask1 sh -c "id"

# -------------------------------
# Step 9: Docker image layers
# -------------------------------
print("\nDocker image layers for nos-flask:1.0:")
!docker history nos-flask:1.0

Folder 'nos-flask' created or already exists.
app.py created with UTF-8 encoding.
requirements.txt created.
Dockerfile created.

Building nos-flask:1.0...

Building nos-flask:1.1 (small change caching demo)...


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 30B 0.1s
#1 transferring dockerfile: 557B 0.3s done
#1 DONE 0.4s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 2.0s

#3 [internal] load .dockerignore
#3 transferring context: 2B 0.0s done
#3 DONE 0.0s

#4 [internal] load build context
#4 DONE 0.0s

#5 [1/6] FROM docker.io/library/python:3.11-slim@sha256:6d98ca198cea726f2c86da2699594339a7b7ff08e49728797b4ed6e3b5c3b62a
#5 resolve docker.io/library/python:3.11-slim@sha256:6d98ca198cea726f2c86da2699594339a7b7ff08e49728797b4ed6e3b5c3b62a
#5 resolve docker.io/library/python:3.11-slim@sha256:6d98ca198cea726f2c86da2699594339a7b7ff08e49728797b4ed6e3b5c3b62a 0.2s done
#5 DONE 0.2s

#4 [internal] load build context
#4 transferring context: 728B 0.5s done
#4 DONE 0.6s

#6 [2/6] WORKDIR /app
#6 CACHED

#7 [3/6] COPY requirements.txt .
#7 DONE 0.2s

#8 [4/6] RUN pip install --no-


Building nos-flask:2.0 (VERSION=2.0)...


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 30B 0.1s
#1 transferring dockerfile: 557B 0.2s done
#1 DONE 0.2s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 0.7s

#3 [internal] load .dockerignore
#3 transferring context: 2B 0.0s done
#3 DONE 0.0s

#4 [internal] load build context
#4 DONE 0.0s

#5 [1/6] FROM docker.io/library/python:3.11-slim@sha256:6d98ca198cea726f2c86da2699594339a7b7ff08e49728797b4ed6e3b5c3b62a
#5 resolve docker.io/library/python:3.11-slim@sha256:6d98ca198cea726f2c86da2699594339a7b7ff08e49728797b4ed6e3b5c3b62a 0.1s done
#5 DONE 0.1s

#4 [internal] load build context
#4 transferring context: 63B 0.0s done
#4 DONE 0.1s

#6 [5/6] COPY app.py .
#6 CACHED

#7 [2/6] WORKDIR /app
#7 CACHED

#8 [3/6] COPY requirements.txt .
#8 CACHED

#9 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#9 CACHED

#10 [6/6] RUN useradd --create-home appuser
#10


Building nos-flask:2.1 (includes /info route)...


...

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    143 100    143   0      0    122      0   00:01   00:01            122
100    143 100    143   0      0    122      0   00:01   00:01            122
100    143 100    143   0      0    122      0   00:01   00:01            122
100    143 100    143   0      0    122      0   00:01   00:01            122



User info in flask1 container:


the input device is not a TTY.  If you are using mintty, try prefixing the command with 'winpty'
the input device is not a TTY.  If you are using mintty, try prefixing the command with 'winpty'


Docker image layers for nos-flask:1.0:


IMAGE          CREATED              CREATED BY                                      SIZE      COMMENT
978940f4a2f3   About a minute ago   CMD ["gunicorn" "--bind" "0.0.0.0:5000" "--wâ€¦   0B        buildkit.dockerfile.v0
<missing>      About a minute ago   USER appuser                                    0B        buildkit.dockerfile.v0
<missing>      About a minute ago   RUN |1 VERSION=1.0 /bin/sh -c useradd --creaâ€¦   69.6kB    buildkit.dockerfile.v0
<missing>      About a minute ago   EXPOSE [5000/tcp]                               0B        buildkit.dockerfile.v0
<missing>      About a minute ago   COPY app.py . # buildkit                        12.3kB    buildkit.dockerfile.v0
<missing>      About a minute ago   RUN |1 VERSION=1.0 /bin/sh -c pip install --â€¦   19.7MB    buildkit.dockerfile.v0
<missing>      About a minute ago   COPY requirements.txt . # buildkit              12.3kB    buildkit.dockerfile.v0
<missing>      9 minutes ago        WORKDIR /app                         